## *A Model Is Only As Good As Its Data*

This has held true since the beginning of machine learning and deep learning. The quality, quantity, diversity and relevance of the data literally make around 80% of the model's performance.  

There is a saying in the data science community: **"Garbage in, garbage out."**   
The model is only as good as the data it has been trained on. If the data is biased, incomplete, or irrelevant, the model will reflect those issues in its predictions and outputs.  
Even the reason it was called a *"model"* is because it only **models** the data it has been trained on.
It is essentially an artifact of the data it has been trained on.

There is even a saying that any model can perform or even outperform a more complex algorithmic model if it's been trained on more and better data.

### Dataset Details

- Total characters: `438,476,740` — not very meaningful for a token-based model. What matters
  is token count, specifically *unique* tokens.
- Total tokens: `100,000,000`  
  Distribution of tokens across datasets:
  - `fineweb`: 60,000,000
  - `wikipedia`: 30,000,000
  - `tinystories`: 10,000,000

All of it is stored as RAW TEXT. There is a single special token:

`<|endoftext|>` — marks whether a sentence/document has ended.

# **How do we turn raw text into a training data for the model?**

#### Section 1: Loading the data

In [15]:
from pathlib import Path
import tiktoken

import torch
from torch.utils.data import Dataset, DataLoader

In [16]:
path_to_data = Path().cwd().parents[1] / 'data' / 'train_data.txt'

In [17]:
with open(path_to_data, 'r', encoding='utf-8') as f:
    raw_text = f.read()

print(f"Total number of characters: {len(raw_text)}")

Total number of characters: 438476740


#### Section 2: Creating Dataset and DataLoader

To create or decide on how the dataset should be you should first ask yourself:
- What am I training the model to do?
- What should it predict?
- What type of training is this — Supervised / Unsupervised / Self-Supervised / Semi-Supervised?
- If it's not unsupervised, what should the labels/targets be?

To answer all of these:  

The model we are training here is a **Generative Model**. Which means it should generate text.  

And as much as there are multiple ways to do it the modern approach now is **Autoregressive** approach.  

**Autoregressive**: is the approach where u predict a word by word/token by token then append the predicted token to the previous set of tokens and predict the next token based on the previous set of tokens and so on and so forth instead of predicting a whole sentence in one go.  

| Question | Answer |
|---|---|
| What am I training the model to do? | Generate text |
| What should it predict? | The next word in a sentence |
| What type of supervision is this? | Self-supervised |
| What should the labels/targets be? | The next word in the sentence |

But why is it self-supervised?  

**Self-Supervised**: is a type of training supervision where the model generates its own labels/targets from the input data. You can think of it as internally it is a supervised learning but the labels/targets are generated from the input data itself No external labels/targets are provided. Note that Self-Supervised and Semi-Supervised are often used interchangeably and some references even consider Self-Supervised as a subset of Semi-Supervised.  

**Semi-Supervised**: is a type of training supervision where the model is trained on a data that has both labeled and unlabeled data. The model is trained on the labeled data and then labels the unlabeled data and then retrains on the newly labeled data.  

To create the dataset we will use a **Sliding Window** approach.

# Sliding Window Approach:

```
"I Ate A Flower Eww It Was Disgusting."

Window Size = 3
Stride = 2

Window 1:
  Input  = [I    | Ate  | A     ]
  Target = [Ate  | A    | Flower]

Window 2 (moves forward by stride = 2):
  Input  = [A     | Flower | Eww]
  Target = [Flower| Eww    | It ]
```

Something to note is that any mathematical model cannot process categorical variables directly,  
so we need to handle that and turn them to numerical numbers.  

That can be done by using a tokenizer, where we will use our tokenizer to **replace** each token (already saved in its own vocabulary) with its **tokenID**.  
Conceptually Similar to label encoding.

GPT Training can be viewed as repeatedly asking:  
***Given the previous tokens, what is the next token?***

<img src="../../images/full_diagram-export.png" width="1000">

***The above image shows the whole process of how the data is prepared for training the model with the sliding window approach.***  
Do note though that it doesnt show the tokenization process and the conversion of tokens to tokenIDs for simplicity.

The data is first tokenized using the tokenizer and stored inside the dataset class
The dataset class then uses the sliding window approach to create the input and target sets and stores them inside 2 lists
`input_ids` and `target_ids`  
then the dataset class starts splitting the data 

### Why `range(0, len(token_ids) - window_size, stride)`?

Stride controls how far the window moves each step. Setting stride = window size means
consecutive windows don't overlap, which helps the model generalize instead of overfitting to
the same span of tokens twice.

We subtract `window_size` from the loop's stop value so the *last* window still has enough
tokens left to build a full input/target pair. Here's why, traced through an example:

```
Sentence:  "I Ate A Flower Eww It Was Disgusting."
Tokens:    [I, Ate, A, Flower, Eww, It, Was, Disgusting, .]
Indices:    0  1    2  3       4    5   6    7            8
len(token_ids) = 9,  window_size = 3,  stride = 2

Loop: range(0, 9 - 3, 2) = range(0, 6, 2) -> i = 0, 2, 4

i=0: Input=[I, Ate, A]        Target=[Ate, A, Flower]
i=2: Input=[A, Flower, Eww]   Target=[Flower, Eww, It]
i=4: Input=[Eww, It, Was]     Target=[It, Was, Disgusting]   <- last window

Next i would be 6, but range(0, 6, 2) stops before reaching 6 (stop is exclusive),
so the loop ends here. That's exactly what we want.
```

Compare that to *not* subtracting `window_size` — `range(0, 9, 2)` would give `i = 0, 2, 4, 6, 8`:

```
i=6: Input=tokens[6:9]  = [Was, Disgusting, .]     (full window, OK)
     Target=tokens[7:10] = [Disgusting, .]         <- only 2 tokens, not 3! Short target.
i=8: Input=tokens[8:11] = [.]                      <- only 1 token. Even worse.
```

Those last windows would be malformed — shorter than `window_size` — which breaks the
assumption that every row in the dataset has equal length. Subtracting `window_size` from the
loop bound is what prevents that.

In [ ]:
# We define the dataset
# It will use the sliding window approach to create inputs and outputs
# With 4 parameters being:
# text, max_window_length, tokenizer, stride

# stride is how how much tokens the window will move in the next iteration
class GPTV0Dataset(Dataset):
    def __init__(self, text, max_window_length, tokenizer, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

        # here we do - max_window_length because 
        # that would make it so each row has equal number of tokens and how it does so is by
        # for example imagine now u do - max_window_length and each time u move a stride
        # if the index > the len - maxwindowlength that would stop the whole loop
        # this is so important because well if we do remove the max_window_length the loop would work properly except
        # for the part at which target = input + 1 token if u remember
        # but u have no more tokens to have so that's the problem
        for i in range(0, len(token_ids) - max_window_length, stride):
            input_window = token_ids[i: i + max_window_length]
            target_window = token_ids[i + 1: i + max_window_length + 1]

            self.input_ids.append(torch.tensor(input_window))
            self.target_ids.append(torch.tensor(target_window))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

### A really important thing about this implementation is 
#### it has a really strong limitation which is that it is too slow and non scalable with large datasets
#### Reason being the algorithm itself is O(n) and that n is the number of tokens in the dataset which is huge imagine a dataset with 1 billion tokens and the algorithm is O(n) so it will take a lot of time to process that dataset and create the input and target sets
##### the problem is this approach literally tokenizes the whole dataset first which is expensive by itself then stores it
#### and to add more to that it then creates the input and target sets by first creating a list of all the input and target sets then storing them in memory which is also expensive and python lists are not memory efficient at all due to the fact that they are dynamic arrays and are resizable so there are lots of overheads which add up 
##### To know the problem more clearly imagine a dataset with 280M tokens, a stride of 256 with a window size of 256
u that would create a whopping 1.09M windows multiply that by 2 since we have input and target sets that would create 2.18M windows lists which is a nightmare and u would have to do all of that before even starting to train the model which is a huge waste of time and resources.

which is why we will cover a more advanced way to do this in the next notebook which is more efficient and scalable with large datasets

In [ ]:
# A VERY IMPORTANT NOTE ABOUT THIS IMPLEMENTATION IS THAT
# IT IS REALLY REALLY SLOW AND NOT PRACTICAL IN MOST CASES 
# the reason it is slow is the algorithm itself at which it constructs the dataset
# u literally tokenize the whole text first which is a bit expensive but not that much
# the problem is the coming step: U TAKE IN THE WHOLE LIST OF TOKENIZED TEXT (which number 1 is a python list that is slow)
# and has some overheads
# second of all U LITERALLY ITERATE THROUGH THE LIST OF TOKENIZED TEXT THEN U 
# 1) CREATE 2 LISTS OBJECTS AGAIN WHICH IS AGAIN VERY SLOW
# CONVERT THEM TO TENSOR TORCHES AND APPEND THEM TO THE LIST OF INDICES AGAIN 
# for a 280M tokens
# and a stride of 256
# that would create a whopping 280M/256 == 1,093,750 LISTS/WINDOWS BEFORE ANYTHING AND DONT FORGET
# U ARE DOING THAT AGAIN FOR THE TARGET SO MULTIPLY THAT BY 2 SO THAT WOULD GIVE U AROUND 2.2M LISTS
# before even the training can start
# and that would also allocate a lot of RAM to store all of that data
# on a normal consumer hardware this isnt feasible + it would take hours before u can even start training

# this is not practical at all
# so instead we will cover extra ways to create a dataset

### DataLoaders are
responsible for creating batches  
A batch is basically a portion of the dataset it is also responsible for things like shuffling the data and what not

In [ ]:
# now we define a function that creates the dataloader for the dataset
def create_dataloaderV0(text, max_window_length, tokenizer, stride, batch_size, shuffle, drop_last, num_workers):
    dataset = GPTV0Dataset(text, max_window_length, tokenizer, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataloader

In [ ]:
tokenizer = tiktoken.get_encoding('gpt2')

dataloader = create_dataloaderV0(raw_text[:10_000], max_window_length=256,      # first 10,000 characters of text
    tokenizer=tokenizer, stride=256, batch_size=1, shuffle=True, drop_last=True, num_workers=0)

In [ ]:
data_iter = iter(dataloader)

first_batch = next(data_iter)
print(f"First batch input shape: {first_batch[0].shape}")
print(f"First batch input: {first_batch[0][0,:10]}")   # first 10 tokens of the first input sequence
print(f"First batch input decoded: {tokenizer.decode(first_batch[0][0,:10].tolist())}")

First batch input shape: torch.Size([1, 256])
First batch input: tensor([  286,   262, 40120,  1015, 18692,    13,   632,   373,   262,  8069])
First batch input decoded:  of the Mojave Desert. It was the 1970


In [ ]:
# this is how many tokens the tokenizer has in its vocabulary already gpt2 bpe
print(f"Tokenizer vocabulary size: {tokenizer.n_vocab}")

Tokenizer vocabulary size: 50257
